In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_C.xpt"
csv_file = DATA_DIR / "BMX_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep completely untouched source copy
source_df = df.copy()


print("=" * 70)
print("NHANES 2003-2004 BMX_C XPT -> CSV")
print("=" * 70)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED BMX_C STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMDRECUF",
    "BMDSUBF",
    "BMDTHICF",
    "BMDLEGF",
    "BMDARMLF",
    "BMDCALFF",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMXLEG",
    "BMILEG",
    "BMXCALF",
    "BMICALF",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXTHICR",
    "BMITHICR",
    "BMXTRI",
    "BMITRI",
    "BMXSUB",
    "BMISUB"
]


if df.columns.tolist() == EXPECTED_COLUMNS:

    print(
        "\nPASS: Column names and order match BMX_C."
    )

else:

    print(
        "\nFAIL: Column names/order differ."
    )

    print("Actual:")
    print(df.columns.tolist())


# ============================================================
# 4. EXAM MEASUREMENT VARIABLES
# ============================================================

#
# IMPORTANT:
#
# THESE MUST REMAIN AS-IS.
#
# Do NOT:
#   round()
#   astype(int)
#   recalculate()
#   derive()
#   change decimal precision
#

MEASUREMENT_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXCALF",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXTHICR",
    "BMXTRI",
    "BMXSUB"
]


# ============================================================
# 5. CODE / FLAG / COMMENT VARIABLES
# ============================================================

#
# These are codebook integer codes.
#
# Examples:
#
# BMDSTATS = 1,2,3,4
# BMIWT    = 1,2,3,4
# BMITRI   = 1,2,3
#
# These can be exported without unnecessary .0.
#

CODE_COLUMNS = [
    "BMDSTATS",
    "BMDRECUF",
    "BMDSUBF",
    "BMDTHICF",
    "BMDLEGF",
    "BMDARMLF",
    "BMDCALFF",
    "BMIWT",
    "BMIRECUM",
    "BMIHEAD",
    "BMIHT",
    "BMILEG",
    "BMICALF",
    "BMIARML",
    "BMIARMC",
    "BMIWAIST",
    "BMITHICR",
    "BMITRI",
    "BMISUB"
]


# ============================================================
# 6. KNOWN PANDAS XPT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

total_zero = int(
    zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{total_zero:,}"
)

print(
    "Tiny artifact values:",
    f"{total_tiny:,}"
)


if total_tiny > 0:

    print("\nVariables containing tiny values:")

    print(
        tiny_counts[
            tiny_counts > 0
        ].sort_values(
            ascending=False
        )
    )

    # IMPORTANT:
    # Unlike HUQ/DEMO/PH, do NOT blindly convert
    # anything in BMX_C to zero.

    raise ValueError(
        "Unexpected tiny-value artifact found in BMX_C. "
        "STOP: validate the affected variable against "
        "the CDC codebook before changing anything."
    )


print(
    "PASS: No tiny zero artifact exists in BMX_C."
)


# ============================================================
# 7. CLEAN SEQN ONLY AS IDENTIFIER
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 8. FORMAT CODE VARIABLES AS INTEGER CODES
# ============================================================

#
# This does NOT touch exam measurements.
#

for col in CODE_COLUMNS:

    if col not in df.columns:
        continue

    values = df[col].dropna()

    if values.empty:
        continue

    fractional = ~np.isclose(
        values,
        np.round(values)
    )

    if fractional.any():

        raise ValueError(
            f"{col} unexpectedly contains fractional "
            f"values. Column was NOT changed."
        )

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: Code/flag/comment variables "
    "formatted as integer codes."
)


# ============================================================
# 9. VERIFY EXAM MEASUREMENTS WERE NOT CHANGED
# ============================================================

print("\n--- MEASUREMENT PRESERVATION CHECK ---")


for col in MEASUREMENT_COLUMNS:

    before = source_df[col].to_numpy()

    after = df[col].to_numpy()

    same = np.allclose(
        before,
        after,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if same:

        print(
            f"PASS: {col:<10} untouched."
        )

    else:

        raise ValueError(
            f"FAIL: {col} was changed."
        )


# ============================================================
# 10. CDC MEASUREMENT EXPECTATIONS
# ============================================================

#
# variable:
# minimum, maximum, valid count, missing count
#

CDC_MEASUREMENTS = {

    "BMXWT": (
        2.4,
        209.1,
        9529,
        114
    ),

    "BMXRECUM": (
        44.3,
        111.2,
        1230,
        8413
    ),

    "BMXHEAD": (
        32.3,
        47.8,
        269,
        9374
    ),

    "BMXHT": (
        79.7,
        204.4,
        8690,
        953
    ),

    "BMXBMI": (
        12.4,
        64.97,
        8687,
        956
    ),

    "BMXLEG": (
        24.7,
        54.0,
        7219,
        2424
    ),

    "BMXCALF": (
        17.5,
        70.5,
        7270,
        2373
    ),

    "BMXARML": (
        9.0,
        48.0,
        9190,
        453
    ),

    "BMXARMC": (
        10.8,
        58.8,
        9187,
        456
    ),

    "BMXWAIST": (
        32.0,
        170.7,
        8397,
        1246
    ),

    "BMXTHICR": (
        28.0,
        94.8,
        7194,
        2449
    ),

    "BMXTRI": (
        2.8,
        44.6,
        8586,
        1057
    ),

    "BMXSUB": (
        3.0,
        41.9,
        8065,
        1578
    )
}


# ============================================================
# 11. CDC RANGE / COUNT VALIDATION
# ============================================================

print("\n--- CDC MEASUREMENT VALIDATION ---")


for col, (
    expected_min,
    expected_max,
    expected_valid,
    expected_missing
) in CDC_MEASUREMENTS.items():

    s = df[col]

    actual_valid = int(
        s.notna().sum()
    )

    actual_missing = int(
        s.isna().sum()
    )

    actual_min = s.min()
    actual_max = s.max()


    passed = (
        actual_valid == expected_valid
        and
        actual_missing == expected_missing
        and
        np.isclose(
            actual_min,
            expected_min
        )
        and
        np.isclose(
            actual_max,
            expected_max
        )
    )


    if passed:

        print(
            f"PASS: {col:<10} "
            f"range={actual_min}-{actual_max}, "
            f"valid={actual_valid:,}, "
            f"missing={actual_missing:,}"
        )

    else:

        print(
            f"FAIL: {col}"
        )

        print(
            "  Actual range:",
            actual_min,
            "to",
            actual_max
        )

        print(
            "  Expected range:",
            expected_min,
            "to",
            expected_max
        )

        print(
            "  Valid:",
            actual_valid,
            "expected",
            expected_valid
        )

        print(
            "  Missing:",
            actual_missing,
            "expected",
            expected_missing
        )


# ============================================================
# 12. CODEBOOK FREQUENCY CHECKS
# ============================================================

CDC_CODES = {

    "BMDSTATS": {
        "codes": {
            1: 7766,
            2: 207,
            3: 1573,
            4: 97
        },
        "missing": 0
    },

    "BMDRECUF": {
        "codes": {
            1: 27
        },
        "missing": 9616
    },

    "BMDSUBF": {
        "codes": {
            1: 1
        },
        "missing": 9642
    },

    "BMDTHICF": {
        "codes": {
            1: 3
        },
        "missing": 9640
    },

    "BMDLEGF": {
        "codes": {
            1: 14
        },
        "missing": 9629
    },

    "BMDARMLF": {
        "codes": {
            1: 5
        },
        "missing": 9638
    },

    "BMDCALFF": {
        "codes": {
            1: 3
        },
        "missing": 9640
    },

    "BMIWT": {
        "codes": {
            1: 24,
            2: 0,
            3: 338,
            4: 17
        },
        "missing": 9264
    },

    "BMIRECUM": {
        "codes": {
            1: 38,
            2: 0,
            3: 0
        },
        "missing": 9605
    },

    "BMIHEAD": {
        "codes": {
            1: 1,
            2: 0,
            3: 0
        },
        "missing": 9642
    },

    "BMIHT": {
        "codes": {
            1: 62,
            2: 0,
            3: 201
        },
        "missing": 9380
    },

    "BMILEG": {
        "codes": {
            1: 322
        },
        "missing": 9321
    },

    "BMICALF": {
        "codes": {
            1: 273
        },
        "missing": 9370
    },

    "BMIARML": {
        "codes": {
            1: 249
        },
        "missing": 9394
    },

    "BMIARMC": {
        "codes": {
            1: 251
        },
        "missing": 9392
    },

    "BMIWAIST": {
        "codes": {
            1: 319
        },
        "missing": 9324
    },

    "BMITHICR": {
        "codes": {
            1: 338
        },
        "missing": 9305
    },

    "BMITRI": {
        "codes": {
            1: 548,
            2: 302,
            3: 0
        },
        "missing": 8793
    },

    "BMISUB": {
        "codes": {
            1: 1283,
            2: 90,
            3: 0
        },
        "missing": 8270
    }
}


print("\n--- CDC CODE FREQUENCY CHECK ---")


for col, spec in CDC_CODES.items():

    passed = True

    for code, expected_count in spec["codes"].items():

        actual_count = int(
            df[col].eq(code).sum()
        )

        if actual_count != expected_count:

            passed = False

            print(
                f"FAIL: {col} code {code}: "
                f"{actual_count} != {expected_count}"
            )


    actual_missing = int(
        df[col].isna().sum()
    )


    if actual_missing != spec["missing"]:

        passed = False

        print(
            f"FAIL: {col} missing: "
            f"{actual_missing} != "
            f"{spec['missing']}"
        )


    if passed:

        print(
            f"PASS: {col}"
        )


# ============================================================
# 13. ROW COUNT CHECK
# ============================================================

print("\n--- ROW COUNT ---")


if len(df) == 9643:

    print(
        "PASS: BMX_C has 9,643 records."
    )

else:

    print(
        "FAIL: Unexpected row count:",
        len(df)
    )


# ============================================================
# 14. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 15. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 16. EXACT EXAM VALUE ROUND-TRIP CHECK
# ============================================================

print("\n--- XPT -> CSV MEASUREMENT FIDELITY ---")


for col in MEASUREMENT_COLUMNS:

    xpt_values = (
        source_df[col]
        .to_numpy()
    )

    csv_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        xpt_values,
        csv_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} "
            "matches XPT exactly."
        )

    else:

        print(
            f"FAIL: {col} changed "
            "during CSV export."
        )

        different = ~np.isclose(
            xpt_values,
            csv_values,
            rtol=0,
            atol=0,
            equal_nan=True
        )

        print(
            pd.DataFrame({
                "XPT":
                    source_df.loc[
                        different,
                        col
                    ],

                "CSV":
                    csv_df.loc[
                        different,
                        col
                    ]
            }).head(20)
        )


# ============================================================
# 17. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 18. RAW SEQN FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    f.readline()

    bad_seqn = 0

    for line in f:

        seqn_text = line.split(",", 1)[0]

        if seqn_text.endswith(".0"):

            bad_seqn += 1


print(
    "SEQN values ending in .0:",
    bad_seqn
)


if bad_seqn == 0:

    print(
        "PASS: SEQN exported without .0."
    )


# ============================================================
# 19. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    csv_df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny artifact values in final CSV:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact exists."
    )


print("\n" + "=" * 70)
print("BMX_C CONVERSION COMPLETE")
print("=" * 70)

NHANES 2003-2004 BMX_C XPT -> CSV
Rows: 9,643
Columns: 33

Variables:
['SEQN', 'BMDSTATS', 'BMDRECUF', 'BMDSUBF', 'BMDTHICF', 'BMDLEGF', 'BMDARMLF', 'BMDCALFF', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMXLEG', 'BMILEG', 'BMXCALF', 'BMICALF', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXTHICR', 'BMITHICR', 'BMXTRI', 'BMITRI', 'BMXSUB', 'BMISUB']

PASS: Column names and order match BMX_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact values: 0
PASS: No tiny zero artifact exists in BMX_C.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: Code/flag/comment variables formatted as integer codes.

--- MEASUREMENT PRESERVATION CHECK ---
PASS: BMXWT      untouched.
PASS: BMXRECUM   untouched.
PASS: BMXHEAD    untouched.
PASS: BMXHT      untouched.
PASS: BMXBMI     untouched.
PASS: BMXLEG     untouched.
PASS: BMXCALF    untouched.
PASS: BMXARML    untouched.
PASS: BMXA